# BitVideo-1.58 Training Notebook

This notebook walks through the complete training pipeline:
1. Prepare your video dataset (extract latents + text embeddings)
2. Configure the model and training hyperparameters
3. Run quantization-aware training (QAT)
4. Monitor progress and evaluate
5. Run inference with trained weights

## 0. Setup

In [1]:
import os
import sys
import json
import torch
import logging
from pathlib import Path

# Add project root to path if running from notebooks/
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

c:\Users\Sachin Kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Memory: 8.5 GB


## 1. Prepare Your Dataset

BitVideo trains on **pre-extracted latents** (not raw video). You need:
- A **Video VAE encoder** (e.g., from Stable Video Diffusion) to compress videos to latent space
- A **Text encoder** (e.g., T5, CLIP) to produce text embeddings

The expected data format:
```
your_data_dir/
  metadata.json
  latents/
    video_0000.pt  # tensor shape [C, T, H, W] (e.g., [4, 16, 32, 32])
    text_0000.pt   # tensor shape [L, D] (e.g., [77, 768])
    video_0001.pt
    text_0001.pt
    ...
```

Run the cell below to prepare your data.

In [2]:
# ============================================================
# CONFIGURE THESE PATHS TO YOUR DATA
# ============================================================

# Your pre-split video clips (4,223 clips already segmented)
CLIPS_DIR = r"G:\My Drive\Antigravity_Production\Training_Output\clips"

# Where processed latents will be saved
DATA_DIR = r"G:\My Drive\Production-Grade-Projects\1-bit\data"

# Model dimensions
LATENT_CHANNELS = 4
NUM_FRAMES = 16         # Frames sampled per clip
LATENT_HEIGHT = 32      # Spatial resolution of latents
LATENT_WIDTH = 32
TEXT_LENGTH = 77
TEXT_DIM = 768

In [3]:
# ============================================================
# OPTION A: Verify existing pre-extracted dataset
# ============================================================

data_path = Path(DATA_DIR)
metadata_file = data_path / "metadata.json"

if metadata_file.exists():
    with open(metadata_file) as f:
        samples = json.load(f)
    print(f"Found {len(samples)} pre-processed samples. Ready to train!")
    s = samples[0]
    v = torch.load(data_path / s['video'], weights_only=True)
    t = torch.load(data_path / s['text'], weights_only=True)
    print(f"  Video latent: {tuple(v.shape)}")
    print(f"  Text embedding: {tuple(t.shape)}")
else:
    print("No pre-processed data found. Run the next cell to process your clips.")
    print(f"Clips source: {CLIPS_DIR}")
    clip_count = len(list(Path(CLIPS_DIR).rglob('*.mp4')))
    print(f"Available clips: {clip_count}")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 649597: character maps to <undefined>

In [ ]:
# ============================================================
# PROCESS YOUR VIDEO CLIPS INTO TRAINING LATENTS
# This reads from your clips folder and creates .pt tensor files
# Run once — skip this cell after your data is prepared
# ============================================================

import cv2
import re
import torch.nn.functional as F

clips_path = Path(CLIPS_DIR)
output_path = Path(DATA_DIR) / 'latents'
output_path.mkdir(parents=True, exist_ok=True)

# Find all clips
all_clips = sorted(clips_path.rglob('*.mp4'))
print(f'Found {len(all_clips)} video clips to process')

metadata = []
errors = []
FRAME_SIZE = (256, 256)

for idx, clip_path in enumerate(all_clips):
    try:
        # Read video frames
        cap = cv2.VideoCapture(str(clip_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            continue
        indices = torch.linspace(0, total_frames - 1, NUM_FRAMES).long().tolist()
        frames = []
        for fi in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (FRAME_SIZE[1], FRAME_SIZE[0]))
                frames.append(torch.from_numpy(frame).permute(2,0,1).float() / 255.0)
            elif frames:
                frames.append(frames[-1].clone())
            else:
                frames.append(torch.zeros(3, FRAME_SIZE[0], FRAME_SIZE[1]))
        cap.release()

        # Stack: [3, T, H, W] and normalize to [-1, 1]
        video = torch.stack(frames, dim=1) * 2.0 - 1.0

        # Downsample to latent resolution: [4, T, 32, 32]
        ds = [F.interpolate(video[:, t:t+1].permute(1,0,2,3), size=(LATENT_HEIGHT, LATENT_WIDTH),
              mode='bilinear', align_corners=False).squeeze(0) for t in range(NUM_FRAMES)]
        latent_3ch = torch.stack(ds, dim=1)  # [3, T, H, W]
        luma = latent_3ch.mean(dim=0, keepdim=True)
        latent = torch.cat([latent_3ch, luma], dim=0)  # [4, T, H, W]

        # Caption from folder name
        caption = re.sub(r'\(?\d+[PKpk]_?(?:HD|60FPS)?\)?', '', clip_path.parent.name)
        caption = re.sub(r'[_\-]+', ' ', caption).strip()
        if not caption: caption = 'a video clip'

        # Pseudo text embedding (seeded from caption for reproducibility)
        seed = hash(caption) % (2**31)
        gen = torch.Generator().manual_seed(seed)
        text_emb = torch.randn(TEXT_LENGTH, TEXT_DIM, generator=gen) * 0.1

        # Save
        v_file = f'latents/video_{idx:04d}.pt'
        t_file = f'latents/text_{idx:04d}.pt'
        torch.save(latent, Path(DATA_DIR) / v_file)
        torch.save(text_emb, Path(DATA_DIR) / t_file)
        metadata.append({'video': v_file, 'text': t_file, 'caption': caption})

        if (idx + 1) % 100 == 0:
            print(f'  Processed {idx + 1}/{len(all_clips)} clips...')

    except Exception as e:
        errors.append(str(e))

# Save metadata
with open(Path(DATA_DIR) / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'\nDone! Processed {len(metadata)} clips ({len(errors)} errors)')
print(f'Output: {DATA_DIR}')

Found 4223 video clips to process
  Processed 100/4223 clips...
  Processed 200/4223 clips...
  Processed 300/4223 clips...
  Processed 400/4223 clips...
  Processed 500/4223 clips...
  Processed 600/4223 clips...
  Processed 700/4223 clips...
  Processed 800/4223 clips...
  Processed 900/4223 clips...
  Processed 1000/4223 clips...
  Processed 1100/4223 clips...
  Processed 1200/4223 clips...
  Processed 1300/4223 clips...
  Processed 1400/4223 clips...
  Processed 1500/4223 clips...
  Processed 1600/4223 clips...
  Processed 1700/4223 clips...
  Processed 1800/4223 clips...
  Processed 1900/4223 clips...
  Processed 2000/4223 clips...
  Processed 2100/4223 clips...
  Processed 2200/4223 clips...
  Processed 2300/4223 clips...
  Processed 2400/4223 clips...
  Processed 2500/4223 clips...
  Processed 2600/4223 clips...
  Processed 2700/4223 clips...
  Processed 2800/4223 clips...
  Processed 2900/4223 clips...
  Processed 3000/4223 clips...
  Processed 3100/4223 clips...
  Processed 32

In [ ]:
# ============================================================
# Create synthetic data for testing (skip if you have real data)
# ============================================================

if not Path(DATA_DIR).exists():
    print("Creating synthetic dataset for demonstration...")
    output_path = Path(DATA_DIR) / "latents"
    output_path.mkdir(parents=True, exist_ok=True)

    NUM_SYNTHETIC_SAMPLES = 100
    metadata = []

    for i in range(NUM_SYNTHETIC_SAMPLES):
        gen = torch.Generator().manual_seed(i)
        video_latent = torch.randn(LATENT_CHANNELS, NUM_FRAMES, LATENT_HEIGHT, LATENT_WIDTH, generator=gen)
        text_emb = torch.randn(TEXT_LENGTH, TEXT_DIM, generator=gen)

        v_path = f"latents/video_{i:04d}.pt"
        t_path = f"latents/text_{i:04d}.pt"
        torch.save(video_latent, Path(DATA_DIR) / v_path)
        torch.save(text_emb, Path(DATA_DIR) / t_path)
        metadata.append({"video": v_path, "text": t_path})

    with open(Path(DATA_DIR) / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Created {NUM_SYNTHETIC_SAMPLES} synthetic samples in {DATA_DIR}")
else:
    print(f"Using existing data in {DATA_DIR}")

## 2. Configure Training

In [ ]:
from bitvideo.training.train_qat_video import TrainingConfig

# ============================================================
# TRAINING CONFIGURATION — ADJUST THESE TO YOUR SETUP
# ============================================================

config = TrainingConfig(
    # Model architecture
    dim=256,                    # Hidden dimension (768 for full scale)
    depth=6,                    # Number of transformer blocks (12 for full)
    num_heads=8,                # Attention heads
    context_dim=TEXT_DIM,       # Must match your text encoder output dim
    in_channels=LATENT_CHANNELS,
    patch_size=(1, 2, 2),       # Temporal, Height, Width patch sizes

    # Data dimensions
    num_frames=NUM_FRAMES,
    height=LATENT_HEIGHT,
    width=LATENT_WIDTH,
    text_length=TEXT_LENGTH,

    # Training hyperparameters
    batch_size=2,               # Reduce if OOM
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=500,
    max_steps=10_000,           # Increase for real training (100k+)
    gradient_accumulation_steps=4,  # Effective batch = batch_size * accum
    max_grad_norm=1.0,

    # Loss
    prediction_type="epsilon",  # or "v_prediction"
    loss_type="mse",
    snr_gamma=5.0,              # Min-SNR weighting (None to disable)

    # Diffusion schedule
    num_train_timesteps=1000,
    beta_schedule="linear",

    # Mixed precision
    mixed_precision="bf16",     # "bf16", "fp16", or "none"

    # Checkpointing
    output_dir="./outputs",
    save_every_steps=2000,
    log_every_steps=50,

    # Hardware
    seed=42,
    num_workers=2,              # DataLoader workers
)

print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
print(f"Training steps: {config.max_steps:,}")
print(f"Mixed precision: {config.mixed_precision}")

Effective batch size: 8
Training steps: 10,000
Mixed precision: bf16


## 3. Initialize Model and Trainer

In [ ]:
from bitvideo.training.train_qat_video import QATTrainer
from bitvideo.training.datasets import VideoTextDataset, create_dataloader

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

# Initialize trainer (creates model, optimizer, scheduler)
trainer = QATTrainer(config)

print(f"\nModel parameters: {trainer.model.parameter_count():,}")
print(f"Device: {trainer.device}")
print(f"Training dtype: {config.mixed_precision}")

2026-07-30 10:42:25,938 Model parameters: 14,283,152



Model parameters: 14,283,152
Device: cuda
Training dtype: bf16


In [ ]:
# Load dataset
dataset = VideoTextDataset(DATA_DIR)
print(f"Dataset size: {len(dataset)} samples")

# Create dataloader
dataloader = create_dataloader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
    drop_last=True,
)

# Verify a batch
sample_batch = next(iter(dataloader))
print(f"Video latent batch: {tuple(sample_batch['video_latent'].shape)}")
print(f"Text embedding batch: {tuple(sample_batch['text_embedding'].shape)}")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 649597: character maps to <undefined>

## 4. Train!

In [ ]:
# ============================================================
# RUN TRAINING
# ============================================================

# Optional: resume from checkpoint
# trainer.load_checkpoint("./outputs/checkpoint-5000.pt")

trainer.train(dataloader)

## 5. Evaluate: Generate Samples

In [ ]:
from bitvideo.pipeline import BitVideoPipeline, DDIMScheduler, DPMPlusPlusScheduler

# Switch to eval mode and pack weights for fast inference
trainer.model.eval()
trainer.model.pack_weights()

# Create inference pipeline
scheduler = DPMPlusPlusScheduler(num_train_steps=1000, prediction_type=config.prediction_type)
pipeline = BitVideoPipeline(trainer.model, scheduler, decoder=None)

# Generate latents
# In practice, encode your text prompt with the same text encoder used for training
test_context = torch.randn(1, TEXT_LENGTH, TEXT_DIM, device=trainer.device, dtype=torch.float16)

with torch.no_grad():
    generated_latents = pipeline(
        test_context,
        num_frames=NUM_FRAMES,
        height=LATENT_HEIGHT,
        width=LATENT_WIDTH,
        num_inference_steps=25,  # DPM++ converges fast
        guidance_scale=7.5,
        negative_context=torch.zeros_like(test_context),
        decode=False,
    )

print(f"Generated latents: {tuple(generated_latents.shape)}")
print(f"Latent stats: mean={generated_latents.mean():.3f}, std={generated_latents.std():.3f}")

## 6. Save Final Model

In [ ]:
# Save the trained model
save_path = Path(config.output_dir) / "final_model.pt"
torch.save({
    "model_state_dict": trainer.model.state_dict(),
    "config": config,
    "global_step": trainer.global_step,
}, save_path)
print(f"Model saved to {save_path}")

# Save config separately for easy loading
from bitvideo.extras import BitVideoConfig, save_config
model_config = BitVideoConfig(
    dim=config.dim,
    depth=config.depth,
    num_heads=config.num_heads,
    context_dim=config.context_dim,
    in_channels=config.in_channels,
    patch_size=config.patch_size,
    prediction_type=config.prediction_type,
)
save_config(model_config, Path(config.output_dir) / "config.json")
print("Config saved.")

## 7. Load and Use Trained Model Later

In [ ]:
# ============================================================
# LOAD A TRAINED MODEL FOR INFERENCE
# ============================================================

from bitvideo.models import VideoDiT
from bitvideo.extras import load_config
from bitvideo.pipeline import BitVideoPipeline, DPMPlusPlusScheduler

# Load config
cfg = load_config("./outputs/config.json")

# Recreate model
model = VideoDiT(
    in_channels=cfg.in_channels,
    dim=cfg.dim,
    depth=cfg.depth,
    num_heads=cfg.num_heads,
    context_dim=cfg.context_dim,
    patch_size=cfg.patch_size,
    qk_norm=True,
    device="cuda",
    dtype=torch.float16,
)

# Load weights
checkpoint = torch.load("./outputs/final_model.pt", map_location="cuda", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
model.pack_weights()  # Enable fast ternary dispatch

# Create pipeline and generate
scheduler = DPMPlusPlusScheduler(prediction_type=cfg.prediction_type)
pipe = BitVideoPipeline(model, scheduler)

print(f"Model loaded from step {checkpoint['global_step']}")
print("Ready for inference!")

## Tips

**Memory issues?**
- Reduce `batch_size` to 1
- Increase `gradient_accumulation_steps` to maintain effective batch size
- Reduce `dim` or `depth` for smaller models
- Use `mixed_precision="bf16"` (most memory efficient)

**Training too slow?**
- Increase `num_workers` for data loading
- Reduce `num_frames` or spatial resolution
- Use smaller model (dim=256, depth=4) for prototyping

**Loss not decreasing?**
- Check learning rate (start with 1e-4)
- Ensure data is normalized (latents should have ~zero mean, ~unit variance)
- Try `prediction_type="v_prediction"` with `snr_gamma=5.0`
- Increase `warmup_steps`

**Want better quality?**
- Train longer (100k+ steps)
- Use knowledge distillation from a full-precision teacher
- Fine-tune with LoRA after initial training
- Use DPM-Solver++ with 25 steps for inference